In [2]:
## Split the dataset
import csv, os, random, sys
from collections import defaultdict
from pathlib import Path
INPUT_ROOT = Path("/kaggle/input")
OUT_DIR = Path("/kaggle/working")
DATASET_DIR_OVERRIDE = "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/plantvillage dataset/color"
CSV_DIR_OVERRIDE = None       
OUTPUT_CSV = OUT_DIR / "plantvillage_splits.csv"
EXCLUDED_LOG = OUT_DIR / "excluded_images_log.csv"
SEED = 42
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15
EXPECTED_CLASSES = 38
IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

def find_keep_lists():
    if CSV_DIR_OVERRIDE:
        return sorted(Path(CSV_DIR_OVERRIDE).glob("*.csv"))
    paths = []
    for dp, dirs, files in os.walk(INPUT_ROOT):
        for n in files:
            if not n.lower().endswith(".csv"):
                continue
            p = Path(dp) / n
            try:
                with open(p, newline="", encoding="utf-8-sig") as f:
                    cols = next(csv.reader(f), [])
            except Exception:
                continue
            cols = [c.strip() for c in cols]
            if "class" in cols and ("path" in cols or "image_path" in cols):
                paths.append(p)
    print("CSV lists found:")
    for p in paths:
        print("  ", p)
    if len(paths) != 3:
        sys.exit(f"ERROR: expected 3 list CSVs, found {len(paths)}. "
                 "Check the dataset with Person 1's files is added under + Add Input.")
    return paths

def find_dataset_dir(classes):
    if DATASET_DIR_OVERRIDE:
        return Path(DATASET_DIR_OVERRIDE)
    classes = set(classes)
    cands = []
    for dp, dirnames, _ in os.walk(INPUT_ROOT):
        hits = len(classes & set(dirnames))
        if hits >= len(classes) // 2:
            cands.append((hits, Path(dp)))
            dirnames[:] = []
    if not cands:
        sys.exit("ERROR: no image folder found. Add the PlantVillage dataset via '+ Add Input'.")
    best = max(h for h, _ in cands)
    top = [p for h, p in cands if h == best]
    if len(top) > 1:
        colour = [p for p in top if "color" in str(p).lower()
                  and not any(w in str(p).lower() for w in ("gray", "grey", "segment"))]
        if len(colour) == 1:
            top = colour
    if len(top) > 1:
        sys.exit("ERROR: several folders found, set DATASET_DIR_OVERRIDE:\n  " + "\n  ".join(map(str, top)))
    return top[0]

def read_keep_lists(list_paths):
    keep = {}
    for lp in list_paths:
        n = 0
        with open(lp, newline="", encoding="utf-8-sig") as f:
            reader = csv.DictReader(f)
            cols = reader.fieldnames or []
            pc = "image_path" if "image_path" in cols else "path" if "path" in cols else None
            if pc is None or "class" not in cols:
                sys.exit(f"ERROR: {lp} needs 'path'/'image_path' and 'class' columns. Found: {cols}")
            for row in reader:
                label = row["class"].strip()
                fname = row[pc].replace("\\", "/").strip().split("/")[-1]
                rel = f"{label}/{fname}"
                if rel in keep:
                    sys.exit(f"ERROR: {rel} appears more than once across the lists.")
                keep[rel] = label
                n += 1
        print(f"  {lp.name}: {n} images")
    return keep

def scan_disk(root):
    found = set()
    for cdir in sorted(d for d in root.iterdir() if d.is_dir()):
        for f in cdir.rglob("*"):
            if f.is_file() and f.suffix.lower() in IMAGE_EXTS:
                found.add(f"{cdir.name}/{f.name}")
    if not found:
        sys.exit(f"ERROR: no images found under {root}")
    return found

def split_counts(n):
    if n >= 3:
        n_val = max(1, round(n * VAL_FRAC))
        n_test = max(1, round(n * (1 - TRAIN_FRAC - VAL_FRAC)))
        return n - n_val - n_test, n_val, n_test
    return n, 0, 0

def make_split(by_class):
    rng = random.Random(SEED)
    rows = []
    for label in sorted(by_class):
        paths = sorted(by_class[label])
        rng.shuffle(paths)
        n_train, n_val, _ = split_counts(len(paths))
        for i, p in enumerate(paths):
            s = "train" if i < n_train else "val" if i < n_train + n_val else "test"
            rows.append((p, label, s))
    return rows

def print_report(rows, classes):
    counts = {c: {"train": 0, "val": 0, "test": 0} for c in classes}
    for _, label, s in rows:
        counts[label][s] += 1
    print(f"\nTotal images (after removing duplicates): {len(rows)}")
    print(f"{'class':50s} {'train':>6} {'val':>6} {'test':>6} {'total':>6}")
    print("-" * 78)
    tot = {"train": 0, "val": 0, "test": 0}
    for c in classes:
        d = counts[c]
        for k in tot: tot[k] += d[k]
        print(f"{c[:50]:50s} {d['train']:6d} {d['val']:6d} {d['test']:6d} {sum(d.values()):6d}")
    print("-" * 78)
    print(f"{'ALL':50s} {tot['train']:6d} {tot['val']:6d} {tot['test']:6d} {len(rows):6d}")
    return counts

def verify(rows, root, classes, counts, excluded):
    print("\n--- VERIFICATION ---")
    ok = True
    def chk(cond, msg):
        nonlocal ok
        print(f"[{'PASS' if cond else 'FAIL'}] {msg}")
        ok &= cond
    paths = [p for p, _, _ in rows]
    chk(all((root / p).is_file() for p in paths), "All image paths exist")
    chk(len(paths) == len(set(paths)), "No image appears more than once (so none in multiple splits)")
    chk(not any(p in excluded for p in paths), "No excluded duplicate is in the CSV")
    chk(all(r[2] in ("train", "val", "test") for r in rows), "Split column contains only train/val/test")
    bad = [c for c in classes if sum(counts[c].values()) >= 3 and min(counts[c].values()) == 0]
    chk(not bad, f"Every class (>= 3 images) is in all 3 splits {bad if bad else ''}")
    tiny = [c for c in classes if 0 < sum(counts[c].values()) < 3]
    if tiny: print(f"[NOTE] Classes with < 3 images cannot be in all splits: {tiny}")
    if len(classes) != EXPECTED_CLASSES:
        print(f"[WARN] Found {len(classes)} classes, expected {EXPECTED_CLASSES}")
    print("\nAll checks passed." if ok else "\nSome checks FAILED.")

def main():
    list_paths = find_keep_lists()
    print("Reading cleaned lists:")
    keep = read_keep_lists(list_paths)
    print(f"Images to keep: {len(keep)}")
    dataset_dir = find_dataset_dir(set(keep.values()))
    print(f"Using image folder: {dataset_dir}")
    on_disk = scan_disk(dataset_dir)
    print(f"Images found on disk: {len(on_disk)}")
    missing = sorted(set(keep) - on_disk)
    if missing:
        sys.exit(f"ERROR: {len(missing)} listed images not in that folder, e.g. {missing[:3]}")
    excluded = on_disk - set(keep)
    print(f"Excluded duplicates (on disk, not in lists): {len(excluded)}")
    with open(EXCLUDED_LOG, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow(["image_path", "label"])
        for p in sorted(excluded): w.writerow([p, p.split("/")[0]])
    by_class = defaultdict(list)
    for rel, label in keep.items(): by_class[label].append(rel)
    classes = sorted(by_class)
    rows = make_split(by_class)
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow(["image_path", "label", "split"])
        w.writerows(sorted(rows))
    print(f"Saved {OUTPUT_CSV}")
    counts = print_report(rows, classes)
    verify(rows, dataset_dir, classes, counts, excluded)

main()

CSV lists found:
   /kaggle/input/datasets/allumalla07/plantvillage-cleaned/part C no dulpicates .csv
   /kaggle/input/datasets/allumalla07/plantvillage-cleaned/plantvillage_color_cleaned.csv
   /kaggle/input/datasets/allumalla07/plantvillage-cleaned/part_b_hashes (1).csv
Reading cleaned lists:
  part C no dulpicates .csv: 18345 images
  plantvillage_color_cleaned.csv: 17613 images
  part_b_hashes (1).csv: 18084 images
Images to keep: 54042
Using image folder: /kaggle/input/datasets/abdallahalidev/plantvillage-dataset/plantvillage dataset/color
Images found on disk: 54305
Excluded duplicates (on disk, not in lists): 263
Saved /kaggle/working/plantvillage_splits.csv

Total images (after removing duplicates): 54042
class                                               train    val   test  total
------------------------------------------------------------------------------
Apple___Apple_scab                                    441     94     95    630
Apple___Black_rot                       

In [3]:
## Add crop and disease columns
import pandas as pd
df = pd.read_csv("/kaggle/working/plantvillage_splits.csv")
parts = df["label"].str.split("___", n=1, expand=True)
df["crop"], df["disease"] = parts[0], parts[1]
df.to_csv("/kaggle/working/plantvillage_splits.csv", index=False)
print(df.head(3)); print(df.crop.nunique(), "crops,", df.disease.nunique(), "disease values")

                                          image_path               label  \
0  Apple___Apple_scab/00075aa8-d81a-4184-8541-b69...  Apple___Apple_scab   
1  Apple___Apple_scab/01a66316-0e98-4d3b-a56f-d78...  Apple___Apple_scab   
2  Apple___Apple_scab/01f3deaa-6143-4b6c-9c22-620...  Apple___Apple_scab   

   split   crop     disease  
0    val  Apple  Apple_scab  
1    val  Apple  Apple_scab  
2  train  Apple  Apple_scab  
14 crops, 21 disease values


In [4]:
## Dataset class and data loaders
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

ROOT = "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/plantvillage dataset/color"
CSV = "/kaggle/working/plantvillage_splits.csv"

# ResNet-50 (ImageNet) normalisation
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = transforms.Compose([            
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([             
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

df = pd.read_csv(CSV)

label_to_idx = {c: i for i, c in enumerate(sorted(df.label.unique()))}
crop_to_idx = {c: i for i, c in enumerate(sorted(df.crop.unique()))}

class PlantVillageDataset(Dataset):
    def __init__(self, df, root, split, transform):
        self.df = df[df.split == split].reset_index(drop=True)
        self.root, self.transform = root, transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(f"{self.root}/{r.image_path}").convert("RGB")
        return (self.transform(img),
                label_to_idx[r.label],   
                crop_to_idx[r.crop])       

train_ds = PlantVillageDataset(df, ROOT, "train", train_tf)
val_ds   = PlantVillageDataset(df, ROOT, "val",   eval_tf)
test_ds  = PlantVillageDataset(df, ROOT, "test",  eval_tf)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# quick checks
imgs, labels, crops = next(iter(train_loader))
print(imgs.shape, labels[:5], crops[:5])        
print(len(train_ds), len(val_ds), len(test_ds)) 
sets = [set(d.df.image_path) for d in (train_ds, val_ds, test_ds)]
print("overlap:", len(sets[0]&sets[1]), len(sets[0]&sets[2]), len(sets[1]&sets[2]))  

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


torch.Size([32, 3, 224, 224]) tensor([ 5, 24, 19, 24, 35]) tensor([ 2, 10,  7, 10, 13])
37827 8107 8108
overlap: 0 0 0
